Część 1

In [11]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    to_timestamp,
    col,
    window,
    avg,
    round as _round,
    sum as _sum,
    min as _min,
    max as _max,
    count,
    desc
)

In [2]:
spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

df = spark.read.json("transactions_10k.jsonl")
df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

print(f"Liczba rekordów: {df.count()}")
df.printSchema()

Spark 4.0.0-preview2 — gotowy
Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



Część 2

In [3]:
# 2.1

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("store")
)
store_summary.show()

+--------+---------+----------+-----------+
|   store|liczba_tx|  suma_PLN|srednia_PLN|
+--------+---------+----------+-----------+
|  Gdańsk|     2498|1021266.35|     408.83|
|  Kraków|     2522|1025896.95|     406.78|
|Warszawa|     2424| 961642.24|     396.72|
| Wrocław|     2556|1002739.21|     392.31|
+--------+---------+----------+-----------+



In [6]:
# 2.2

category_summary = (
    df.groupBy("category")
    .agg(
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(_min("amount"), 2).alias("min_PLN"),
        _round(_max("amount"), 2).alias("max_PLN")
    )
    .orderBy("category")
)

category_summary.show()

+-----------+----------+-------+-------+
|   category|  suma_PLN|min_PLN|max_PLN|
+-----------+----------+-------+-------+
|elektronika|1520770.69|    9.0| 9999.0|
|    książki| 851382.08|    5.0|9107.25|
|     odzież| 849877.55|    5.0|9696.63|
|    żywność| 789514.43|    5.0|6916.92|
+-----------+----------+-------+-------+



Część 3

In [7]:
# 3.1

hourly = (
    df.groupBy(window("timestamp", "1 hour"))    # okno 1-godzinne
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)
hourly.show(truncate=False)

+------------------------------------------+---------+----------+
|window                                    |liczba_tx|suma_PLN  |
+------------------------------------------+---------+----------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|3150     |1241911.3 |
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|4661     |1896230.21|
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|2189     |873403.24 |
+------------------------------------------+---------+----------+



In [8]:
(
    hourly
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .show(truncate=False)
)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
+-------------------+-------------------+---------+----------+



In [10]:
# 3.2

(
    df.groupBy(window("timestamp", "30 minutes"), "store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN")
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "store",
        "liczba_tx",
        "suma_PLN"
    )
    .orderBy("od", "store")
).show(truncate=False)

+-------------------+-------------------+--------+---------+---------+
|od                 |do                 |store   |liczba_tx|suma_PLN |
+-------------------+-------------------+--------+---------+---------+
|2026-04-12 08:00:00|2026-04-12 08:30:00|Gdańsk  |252      |93391.22 |
|2026-04-12 08:00:00|2026-04-12 08:30:00|Kraków  |289      |117786.42|
|2026-04-12 08:00:00|2026-04-12 08:30:00|Warszawa|275      |88441.58 |
|2026-04-12 08:00:00|2026-04-12 08:30:00|Wrocław |296      |111540.59|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Gdańsk  |514      |209187.85|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Kraków  |532      |223541.41|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Warszawa|490      |182435.06|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Wrocław |502      |215587.17|
|2026-04-12 09:00:00|2026-04-12 09:30:00|Gdańsk  |619      |253364.95|
|2026-04-12 09:00:00|2026-04-12 09:30:00|Kraków  |590      |224358.03|
|2026-04-12 09:00:00|2026-04-12 09:30:00|Warszawa|584      |214573.66|
|2026-

In [16]:
# 3.3

(
    df.filter(col("store") == "Kraków")
    .groupBy(window("timestamp", "1 hour"))
    .agg(
        _round(_sum("amount"), 2).alias("suma_PLN")
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "suma_PLN"
    )
    .orderBy(desc("suma_PLN"))
    .limit(1)
).show(truncate=False)

+-------------------+-------------------+---------+
|od                 |do                 |suma_PLN |
+-------------------+-------------------+---------+
|2026-04-12 09:00:00|2026-04-12 10:00:00|483309.86|
+-------------------+-------------------+---------+



Część 4

In [17]:
# 4.1

sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))  # szerokość 1h, krok 30min
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od")
)
sliding.show(truncate=False)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 07:30:00|2026-04-12 08:30:00|1112     |411159.81 |
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 08:30:00|2026-04-12 09:30:00|4443     |1753033.6 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 09:30:00|2026-04-12 10:30:00|3696     |1557641.39|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
|2026-04-12 10:30:00|2026-04-12 11:30:00|749      |289709.95 |
+-------------------+-------------------+---------+----------+



In [19]:
# 4.2

tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)
sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)
print(f"Tumbling (1h):          {tumbling_rows} okien")
print(f"Sliding  (1h / 30min):  {sliding_rows} okien")

# Odpowiedz w komentarzu: dlaczego sliding ma więcej wierszy?
# TWOJA ODPOWIEDŹ:

# Sliding windows nakładają się na siebie (co 30 min tworzy się okno 1h).
# W efekcie liczba wygenerowanych wierszy jest większa niż w tumbling,
# gdzie okna są rozłączne i każde "pół godziny" należy tylko do "jednej godziny".
# W skrócie - się nakładają i "duplikują"

Tumbling (1h):          3 okien
Sliding  (1h / 30min):  7 okien


Część 5

In [ ]:
# Odpowiedz na pytania w komentarzach:

# 1. Ile transakcji jest w oknie 09:00–10:00?
#    Sprawdź w wyniku zadania 3.1.
#    ODPOWIEDŹ: 4661

# 2. Jaka jest różnica między groupBy("store") a groupBy(window(...), "store")?
#    ODPOWIEDŹ: groupBy("store") - agreguje transakcje po sklepie z całego zbioru ignorując godzinę transakcji
#               groupBy(window(...), "store") - agreguje po sklepie oraz po oknie czasowym - czyli agregacja per sklep per okno

# 3. W oknie sliding 1h/30min — ile okien zawiera transakcje z godziny 09:30?
#    Wskazówka: narysuj oś czasu.
#    ODPOWIEDŹ: 2 okna - 08:30-09:30, 09:00-10:00

In [ ]:
Zadania domowe

In [31]:
def avg_amount_by_window(df, store_name, window_duration):
    return (
        df.filter(col("store") == store_name)
        .groupBy(window("timestamp", window_duration))
        .agg(
            _round(avg("amount"), 2).alias("srednia")
        )
        .select(
            col("window.start").alias("od"),
            col("window.end").alias("do"),
            "srednia"
        )
        .orderBy("od")
        .limit(1)
    )

avg_amount_by_window(df, "Gdańsk", "1 hour").show(truncate=False)

+-------------------+-------------------+-------+
|od                 |do                 |srednia|
+-------------------+-------------------+-------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|395.01 |
+-------------------+-------------------+-------+



In [20]:
def count_transactions_by_category(df, start_time, end_time):
    return (
        df.filter(
            (col("timestamp") >= start_time) &
            (col("timestamp") < end_time)
        )
        .groupBy("category")
        .agg(
            count("tx_id").alias("liczba_tx")
        )
        .orderBy("category")
    )

count_transactions_by_category(df,"2026-04-12 09:00:00","2026-04-12 09:30:00").show(truncate=False)

+-----------+---------+
|category   |liczba_tx|
+-----------+---------+
|elektronika|611      |
|książki    |622      |
|odzież     |605      |
|żywność    |567      |
+-----------+---------+



In [33]:
def peak_transactions_window(df, window_duration):
    return (
        df.groupBy(window("timestamp", window_duration))
        .agg(
            count("tx_id").alias("liczba_tx")
        )
        .select(
            col("window.start").alias("od"),
            col("window.end").alias("do"),
            "liczba_tx"
        )
        .orderBy(col("liczba_tx").desc())
        .limit(1)
    )
    
peak_transactions_window(df, "15 minutes").show(truncate=False)

+-------------------+-------------------+---------+
|od                 |do                 |liczba_tx|
+-------------------+-------------------+---------+
|2026-04-12 09:15:00|2026-04-12 09:30:00|1234     |
+-------------------+-------------------+---------+



In [21]:
spark.stop()